# Aprendizaje por Refuerzo - Deep Q-learning (DQN)
### Versión actualizada con librerías modernas

**Librerías utilizadas:**
- `gymnasium` (sucesor de OpenAI Gym)
- `stable-baselines3` (sucesor de keras-rl)
- `pytorch` (backend de SB3)

**Instalación:**
```bash
pip install gymnasium
pip install stable-baselines3[extra]
pip install gymnasium[atari]
pip install autorom[accept-rom-license]
pip install torch
```

In [3]:
!pip install stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 23.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 30.0 MB/s eta 0:00:00


In [4]:
# Verificar versiones instaladas
!pip show gymnasium stable-baselines3 torch | grep -E 'Name|Version'

Name: gymnasium
Version: 1.2.3
Name: stable_baselines3
Version: 2.7.1
Name: torch
Version: 2.9.0


---
## Parte 1: CartPole con DQN

Entorno clásico de control. El objetivo es mantener el palo en equilibrio.

**Documentación:**
- https://gymnasium.farama.org/
- https://stable-baselines3.readthedocs.io/

### DQN Pseudo-código

<img src="./IMG/dqn.png" />

In [5]:
import numpy as np
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

2026-02-13 16:40:58.200526: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-13 16:40:58.297276: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-13 16:41:00.454761: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [6]:
ENV_NAME = 'CartPole-v1'  # v1 es la versión actual (v0 está deprecada)

# Crear el entorno
env = gym.make(ENV_NAME)

# Número de acciones posibles
nb_actions = env.action_space.n
print(f'Número de acciones: {nb_actions}')

Número de acciones: 2


In [7]:
# Espacio de observaciones
print(f'Observation space: {env.observation_space}')
print(f'Shape: {env.observation_space.shape}')

Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Shape: (4,)


### Configuración del agente DQN

En Stable-Baselines3, la **memoria** (ReplayBuffer) y la **política** (epsilon-greedy) se configuran directamente en el constructor del agente:

| keras-rl2 | Stable-Baselines3 |
|---|---|
| `SequentialMemory(limit=50000)` | `buffer_size=50000` |
| `BoltzmannQPolicy()` | `exploration_fraction` + `exploration_final_eps` |
| `nb_steps_warmup=10` | `learning_starts=10` |
| `target_model_update=1e-2` | `tau=0.01` |
| `dqn.compile(Adam(lr=1e-3))` | `learning_rate=1e-3` |

In [8]:
# Crear el agente DQN
# 'MlpPolicy' usa una red neuronal densa (equivalente al modelo Sequential con Dense layers del original)
model = DQN(
    policy='MlpPolicy',        # red neuronal densa para observaciones vectoriales
    env=env,
    learning_rate=1e-3,        # equivalente a Adam(lr=1e-3)
    buffer_size=50000,         # equivalente a SequentialMemory(limit=50000)
    learning_starts=10,        # equivalente a nb_steps_warmup=10
    batch_size=32,
    tau=0.01,                  # equivalente a target_model_update=1e-2
    gamma=0.99,                # factor de descuento
    exploration_fraction=0.1,  # fracción del entrenamiento con exploración
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    verbose=1
)

# Ver la arquitectura de la red neuronal interna
print(model.policy)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
DQNPolicy(
  (q_net): QNetwork(
    (features_extractor): FlattenExtractor(
      (flatten): Flatten(start_dim=1, end_dim=-1)
    )
    (q_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=2, bias=True)
    )
  )
  (q_net_target): QNetwork(
    (features_extractor): FlattenExtractor(
      (flatten): Flatten(start_dim=1, end_dim=-1)
    )
    (q_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=2, bias=True)
    )
  )
)


In [9]:
# Entrenar el agente
# equivalente a dqn.fit(env, nb_steps=50000)
model.learn(
    total_timesteps=50000,
    log_interval=10         # mostrar métricas cada 10 episodios
)

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21       |
|    ep_rew_mean      | 21       |
|    exploration_rate | 0.96     |
| time/               |          |
|    episodes         | 10       |
|    fps              | 219      |
|    time_elapsed     | 0        |
|    total_timesteps  | 210      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.0333   |
|    n_updates        | 50       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 19.5     |
|    ep_rew_mean      | 19.5     |
|    exploration_rate | 0.926    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 301      |
|    time_elapsed     | 1        |
|    total_timesteps  | 390      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.00244  |
|    n_updates      

In [10]:
# Guardar el modelo entrenado
# equivalente a dqn.save_weights('dqn_CartPole_weights.h5f')
model.save('dqn_cartpole_model')
print('Modelo guardado como dqn_cartpole_model.zip')

Modelo guardado como dqn_cartpole_model.zip


In [11]:
# Evaluar el agente entrenado
# equivalente a dqn.test(env, nb_episodes=10)
mean_reward, std_reward = evaluate_policy(
    model,
    env,
    n_eval_episodes=10,
    deterministic=True
)
print(f'Recompensa media: {mean_reward:.2f} ± {std_reward:.2f}')

Recompensa media: 9.80 ± 0.40


/home/jordi/Documentos/Ribera/Curso_25_26/CEIABD/CEIABD_25_26/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


In [ ]:
# Creamos un nuevo entorno para probar el modelo cargado y renderizarlo
env = gym.make(ENV_NAME, render_mode='rgb_array')
# Cargar un modelo guardado y probarlo
# equivalente a dqn.load_weights(...) + dqn.test(...)
loaded_model = DQN.load('dqn_cartpole_model', env=env)

# Preparamos el array de observaciones para el modelo
frames = []

obs, info = env.reset()
for _ in range(1000):  # ejecutar el agente durante 1000 pasos
    frames.append(env.render())  # renderizar el entorno y guardar el frame
    action, _states = loaded_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        obs, info = env.reset()

env.close()

# Guardamos los frames como un video
import imageio

imageio.mimwrite('dqn_cartpole_video.mp4', frames, fps=30)
print('Video guardado como dqn_cartpole_video.mp4')

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/jordi/Documentos/Ribera/Curso_25_26/CEIABD/CEIABD_25_26/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Video guardado como dqn_cartpole_video.mp4


In [17]:
import base64
from IPython.display import HTML, display
# Mostrar el vídeo inline en Jupyter
with open('dqn_cartpole_video.mp4', 'rb') as f:
    video_data = base64.b64encode(f.read()).decode()

display(HTML(f"""
    <video width="400" controls autoplay loop>
        <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
    </video>
"""))

---
## Parte 2: Atari Breakout con DQN

Entorno Atari con entrada de píxeles. Requiere una CNN como arquitectura.

**Cambios respecto al original:**
- `BreakoutDeterministic-v4` → `ALE/Breakout-v5` (nomenclatura moderna)
- `AtariProcessor` → wrappers integrados en SB3 (`AtariWrapper`)
- `Convolution2D` + `Permute` → `CnnPolicy` (SB3 lo gestiona automáticamente)
- `LinearAnnealedPolicy(EpsGreedyQPolicy())` → `exploration_fraction` en DQN

In [20]:
!pip install gymnasium[atari]
!pip install autorom[accept-rom-license]

In [3]:
import gymnasium as gym
import ale_py
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

In [4]:
env_name = 'ALE/Breakout-v5'  # equivalente a BreakoutDeterministic-v4

# make_atari_env aplica automáticamente los wrappers equivalentes al AtariProcessor original:
#   - Redimensiona frames a 84x84
#   - Convierte a escala de grises
#   - Normaliza píxeles a [0, 1]
#   - Clip de recompensas a [-1, 1]
env = make_atari_env(env_name, n_envs=1, seed=123)

# VecFrameStack apila 4 frames consecutivos (equivalente a WINDOW_LENGTH=4)
env = VecFrameStack(env, n_stack=4)

print(f'Observation space: {env.observation_space}')
print(f'Action space: {env.action_space}')

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Observation space: Box(0, 255, (84, 84, 4), uint8)
Action space: Discrete(4)


In [5]:
# Número de acciones
nb_actions = env.action_space.n
print(f'Número de acciones: {nb_actions}')

Número de acciones: 4


### Equivalencias de configuración Atari

| keras-rl2 (original) | Stable-Baselines3 (nuevo) |
|---|---|
| `AtariProcessor` | `make_atari_env()` automático |
| `SequentialMemory(limit=1000000)` | `buffer_size=100000` |
| `WINDOW_LENGTH = 4` | `VecFrameStack(env, n_stack=4)` |
| `LinearAnnealedPolicy(EpsGreedyQPolicy(), nb_steps=1000000)` | `exploration_fraction=0.1` |
| `nb_steps_warmup=50000` | `learning_starts=50000` |
| `target_model_update=10000` | `target_update_interval=10000` |
| `train_interval=20` | `train_freq=4` |
| `Convolution2D + Permute` | `CnnPolicy` (automático) |

In [6]:
# Configurar callbacks para guardar checkpoints y logs
# equivalente a ModelIntervalCheckpoint + FileLogger del original

checkpoint_callback = CheckpointCallback(
    save_freq=250000,
    save_path='./checkpoints/',
    name_prefix=f'dqn_{env_name.replace("/", "_")}'
)

eval_env = make_atari_env(env_name, n_envs=1, seed=42)
eval_env = VecFrameStack(eval_env, n_stack=4)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path='./best_model/',
    log_path='./logs/',
    eval_freq=50000,
    n_eval_episodes=5
)

In [8]:
# Crear el agente DQN para Atari
# 'CnnPolicy' usa la arquitectura CNN equivalente al modelo Convolution2D del original
model = DQN(
    policy='CnnPolicy',             # CNN equivalente al modelo original con Convolution2D
    env=env,
    learning_rate=0.00025,          # equivalente a Adam(lr=0.00025)
    buffer_size=100000,             # reducido para memoria RAM (original: 1000000)
    learning_starts=50000,          # equivalente a nb_steps_warmup=50000
    batch_size=32,
    tau=1.0,
    gamma=0.99,
    target_update_interval=10000,   # equivalente a target_model_update=10000
    train_freq=4,                   # equivalente a train_interval
    exploration_fraction=0.1,       # equivalente a LinearAnnealedPolicy(...nb_steps=1000000)
    exploration_initial_eps=1.0,    # equivalente a value_max=1.
    exploration_final_eps=0.01,     # equivalente a value_min=.1
    optimize_memory_usage=False,     # optimización de memoria para buffer grande
    verbose=1
)

print(model.policy)

Using cuda device
Wrapping the env in a VecTransposeImage.
CnnPolicy(
  (q_net): QNetwork(
    (features_extractor): NatureCNN(
      (cnn): Sequential(
        (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
        (1): ReLU()
        (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
        (3): ReLU()
        (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
        (5): ReLU()
        (6): Flatten(start_dim=1, end_dim=-1)
      )
      (linear): Sequential(
        (0): Linear(in_features=3136, out_features=512, bias=True)
        (1): ReLU()
      )
    )
    (q_net): Sequential(
      (0): Linear(in_features=512, out_features=4, bias=True)
    )
  )
  (q_net_target): QNetwork(
    (features_extractor): NatureCNN(
      (cnn): Sequential(
        (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
        (1): ReLU()
        (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
        (3): ReLU()
        (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
 

In [10]:
# Entrenar el agente
# equivalente a dqn.fit(env, nb_steps=1750000, callbacks=callbacks)
weights_filename = f'dqn_{env_name.replace("/", "_")}_final'

model.learn(
    # total_timesteps=1750000,
    total_timesteps=100000,  # reducido para pruebas rápidas (original: 1750000)
    callback=[checkpoint_callback, eval_callback],
    log_interval=10000
)

# Guardar el modelo final
# equivalente a dqn.save_weights(weights_filename)
model.save(weights_filename)
print(f'Modelo guardado: {weights_filename}.zip')

Eval num_timesteps=46636, episode_reward=22.20 +/- 10.63
Episode length: 777.60 +/- 247.10
----------------------------------
| eval/               |          |
|    mean_ep_length   | 778      |
|    mean_reward      | 22.2     |
| rollout/            |          |
|    exploration_rate | 0.01     |
| time/               |          |
|    total_timesteps  | 46636    |
----------------------------------
Eval num_timesteps=96636, episode_reward=26.20 +/- 7.03
Episode length: 880.00 +/- 154.34
----------------------------------
| eval/               |          |
|    mean_ep_length   | 880      |
|    mean_reward      | 26.2     |
| rollout/            |          |
|    exploration_rate | 0.01     |
| time/               |          |
|    total_timesteps  | 96636    |
| train/              |          |
|    learning_rate    | 0.00025  |
|    loss             | 0.00616  |
|    n_updates        | 74999    |
----------------------------------
Modelo guardado: dqn_ALE_Breakout-v5_final.zip


#### Información de salida del entrenamiento

1. **Métricas de Evaluación** (Sección eval/)
Estos datos provienen de episodios de prueba independientes donde el agente no explora (usa una política determinista) para medir su rendimiento real.
- ***num_timesteps*** = 46636: Es el momento exacto del entrenamiento en el que se ha realizado la evaluación. Lleva acumulados unos 46 mil pasos de interacción con el entorno.
- ***episode_reward*** = 22.20 +/- 10.63: Esta es la métrica reina. El agente obtiene una recompensa media de 22.20. El +/- 10.63 es la desviación estándar; una desviación alta indica que el agente todavía es inconsistente (a veces lo hace muy bien y otras muy mal).
- ***Episode length*** (mean_ep_length) = 777.60: Indica cuánto duran los episodios de media. En muchos entornos de RL, si la duración aumenta, suele significar que el agente está aguantando más tiempo sin "morir" o fallar.   

2. **Estado de la Exploración** (Sección rollout/)
- exploration_rate = 0.01: Este es el valor de $\epsilon$ (epsilon) en la estrategia Epsilon-Greedy.
    - Como se ha definido un `exploration_final_eps=0.01`, significa que el agente ya ha alcanzado el nivel mínimo de exploración.
    - Ojo aquí: Solo lleva 46k pasos y ya está al mínimo (1%). Esto sucede porque, aunque la exploration_fraction es 0.1 (lo que sobre 1.7M pasos serían 170k pasos de bajada), SB3 calcula la caída lineal desde el principio. Al ritmo que va, el agente ha dejado de explorar muy pronto para un entrenamiento tan largo.
3. **Rendimiento Temporal** (Sección time/)
- ***total_timesteps*** = 46636: El contador global de pasos que el agente ha dado en el entorno desde que se ejecutó `model.learn`.

### Modificaciones clave para reducir drásticamente el tiempo de entrenamiento, priorizando la velocidad sobre la convergencia perfecta:

1. Reducir drásticamente los pasos totales
    -El parámetro `total_timesteps` es el que más influye. Si se está en fase de pruebas o desarrollo:

    -Modificación: Bajar de 1,750,000 a 100,000 o 200,000.

    -Razón: permitirá ver si el código funciona y si la curva de aprendizaje (en el eval_callback) tiene sentido antes de comprometer horas de cómputo.

2. Ajustar el "Warmup" (Calentamiento)
    -El valor de `learning_starts=50,000`.

    -Modificación: Reducir a 1,000 o 5,000.

    -Razón: El modelo está pasando los primeros 50,000 pasos realizando acciones puramente aleatorias para llenar el buffer sin aprender nada. Al bajarlo, el entrenamiento (la actualización de pesos) empieza mucho antes.

3. Aumentar la frecuencia de entrenamiento
    -El valor `train_freq=4`.

    -Modificación: Subir a 16 o 32.

    -Razón: En estas condiciones, el modelo realiza un paso de backpropagation cada 4 pasos en el entorno. Si lo subes, entrenará menos veces por cada interacción, lo que acelera el reloj enormemente a costa de una actualización menos frecuente de la política.

4. Cambiar la Arquitectura (Si es posible)
    -Si el entorno no es visualmente complejo (como un juego de Atari con píxeles crudos):

    -Modificación: Cambiar policy='CnnPolicy' por policy='MlpPolicy'.

    -Razón: Las capas densas (MLP) son órdenes de magnitud más rápidas de entrenar que las convolucionales. Si el `observation_space` permite ser aplanado (flatten), esta es la mejora de velocidad más grande.

In [11]:
# Cargar y evaluar el modelo
# equivalente a dqn.load_weights(...) + dqn.test(env, nb_episodes=10)
loaded_model = DQN.load(weights_filename, env=env)

mean_reward, std_reward = evaluate_policy(
    loaded_model,
    env,
    n_eval_episodes=10,
    deterministic=False  # False para usar algo de exploración en test (equivalente a value_test=.05)
)
print(f'Recompensa media en test: {mean_reward:.2f} ± {std_reward:.2f}')

Wrapping the env in a VecTransposeImage.
Recompensa media en test: 27.20 ± 10.51


In [12]:
# Visualizar el agente jugando
# equivalente a dqn.test(env, nb_episodes=5, visualize=True)
render_env = make_atari_env(env_name, n_envs=1, seed=0, env_kwargs={'render_mode': 'human'})
render_env = VecFrameStack(render_env, n_stack=4)

obs = render_env.reset()
for _ in range(5000):
    action, _states = loaded_model.predict(obs, deterministic=False)
    obs, rewards, dones, info = render_env.step(action)

render_env.close()

: 

---
## Notas sobre la migración

**Principales cambios:**

1. **`gym` → `gymnasium`**: API idéntica pero con una diferencia importante: `env.step()` ahora devuelve 5 valores (`obs, reward, terminated, truncated, info`) en lugar de 4.

2. **`keras-rl2` → `stable-baselines3`**: Ya no es necesario definir el modelo Keras manualmente. SB3 usa `MlpPolicy` o `CnnPolicy` que construyen la red automáticamente.

3. **`AtariProcessor` → `make_atari_env()`**: Los wrappers de preprocesamiento están integrados y se aplican automáticamente.

4. **`SequentialMemory` → `buffer_size`**: El replay buffer está incorporado en el agente DQN.

5. **`BoltzmannQPolicy` / `LinearAnnealedPolicy`**: SB3 usa epsilon-greedy por defecto. Si necesitas Boltzmann, se puede implementar mediante una subclase personalizada.

6. **`env.seed()` → deprecado**: La semilla ahora se pasa directamente a `gym.make()` o `make_atari_env()`.